# ReaderAdapt-Liver on LiTS17 with one pretrained model: NVIDIA VISTA3D (Colab, GPU)

**Runtime -> Change runtime type -> GPU** (T4 works; L4 / A100 is faster).

One Colab secret (key icon on the left, *Notebook access* on):

| Secret | Why | Where to get it |
|---|---|---|
| `KAGGLE_API_TOKEN` (or `KAGGLE_USERNAME` + `KAGGLE_KEY`) | download the two LiTS17 datasets | kaggle.com -> Settings -> API -> Create New Token |

The segmentation model, **NVIDIA VISTA3D** (MONAI), is downloaded from Hugging Face (`MONAI/VISTA3D-HF`,
~870 MB) without a token and used as published - **nothing is trained**. It segments the liver and hepatic
tumours in 3D. Its weights are for non-commercial research use (NVIDIA NCLS v1).

Pipeline: download LiTS (both Kaggle parts) -> calib / test split -> fetch VISTA3D -> segment + measure every
CT -> calibration -> agentic first read + reports -> reader experiment -> figures -> **website**.

Note: VISTA3D's training data include MSD Task 03, built from the LiTS training volumes, so its segmentation
quality on LiTS is optimistic (not a held-out evaluation).

Rough times on a T4: download 10-15 min; VISTA3D about 1-2 min per CT with the flip pass (~2-3 h for all 131).
Set `QUICK = True` below for a ~40-case first look without the flip pass.

In [ ]:
# 1. Code + dependencies
REPO_URL = "https://github.com/AIVIETNAM-AIO-khaisiubeo/ReaderAdapt-Liver.git"
BRANCH = "vista3d-single-model"   # use "main" once the pull requests are merged
import os
if not os.path.exists("ReaderAdapt-Liver"):
    !git clone -q -b $BRANCH $REPO_URL
%cd ReaderAdapt-Liver
!git pull -q
!pip install -q -r requirements.txt -r requirements-deep.txt

In [ ]:
# 2. Secrets -> environment (KAGGLE_API_TOKEN, or KAGGLE_USERNAME + KAGGLE_KEY; HF_TOKEN optional)
import os
from google.colab import userdata
for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"):
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = v
    except Exception:
        pass
print({k: (k in os.environ) for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY")})
import torch
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime!")

In [ ]:
# 3. Run configuration
import yaml, torch
QUICK = False                     # True: proportional ~40-case subset, no flip pass, for a fast first look
cfg = yaml.safe_load(open("configs/lits.yaml"))
big_gpu = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 30e9
cfg["segmenter"]["sw_batch_size"] = 4 if big_gpu else 1
if QUICK:
    cfg["run_name"], cfg["data"]["max_cases"], cfg["inference"]["tta_flip"] = "lits_quick", 40, False
yaml.safe_dump(cfg, open("configs/lits_colab.yaml", "w"), sort_keys=False)
RUN = f"outputs/{cfg['run_name']}"
print("run:", RUN, "| segmenter: NVIDIA VISTA3D (pretrained) | flip pass:", cfg["inference"]["tta_flip"])

In [ ]:
# 4. LiTS17 from Kaggle (part 1 = segmentations + volume_pt1-5, part 2 = the remaining volumes), then the split
!python -m radagent download --config configs/lits_colab.yaml
!python -m radagent data --config configs/lits_colab.yaml

In [ ]:
# 5. Fetch the pretrained NVIDIA VISTA3D weights (~870 MB, cached) and write the model card - no training
!python -m radagent model --config configs/lits_colab.yaml

In [ ]:
# 6. Segment + measure every CT with VISTA3D (liver + hepatic tumour; CT and its mirror image)
!python -m radagent infer --config configs/lits_colab.yaml

In [ ]:
# 7. Calibration, agentic first read + reports, reader experiment, figures
!python -m radagent calibrate --config configs/lits_colab.yaml
!python -m radagent first-read --config configs/lits_colab.yaml
!python -m radagent simulate --config configs/lits_colab.yaml
!python -m radagent figures --config configs/lits_colab.yaml

In [ ]:
# 8. One agent trace and its report
!python -m radagent agent --config configs/lits_colab.yaml
import glob
print(open(sorted(glob.glob(f"{RUN}/reports/*.txt"))[0]).read())

In [ ]:
from IPython.display import Image, display
for f in ("calibration_reliability.png", "segmentation_quality.png", "policy_comparison.png"):
    if os.path.exists(f"{RUN}/figures/{f}"):
        display(Image(f"{RUN}/figures/{f}", width=900))

## 9. Open the website

**Option A - straight from Colab (everything works, including *Recompute from the CT file*, because the CTs, the
GPU and the model are here).** The cell starts the website and prints a public `trycloudflare.com` link (a free
Cloudflare quick tunnel; anyone with the link can open it while the cell runs - stop the cell to close it).

In [ ]:
import subprocess, re, time
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
app = subprocess.Popen(["python", "-m", "streamlit", "run", "app/streamlit_app.py", "--server.port", "8501",
                        "--server.headless", "true", "--", "--run", RUN],
                       stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("Website:", m.group(0))
        break
try:
    while app.poll() is None:
        time.sleep(30)
except KeyboardInterrupt:
    pass
finally:
    tunnel.terminate(); app.terminate()

**Option B - on your own computer.** Download the run, unzip it into the `outputs/` folder of your local copy of
the repository and start `python -m streamlit run app/streamlit_app.py -- --run outputs/lits`. Everything except
*Recompute from the CT file* works without a GPU (recompute needs the LiTS files, PyTorch, MONAI and the VISTA3D
weights; a GPU is recommended).

In [ ]:
!cd outputs && zip -qr ../lits_run.zip {cfg['run_name']}
!ls -lh lits_run.zip
from google.colab import files
files.download("lits_run.zip")
# Optional: keep a copy in Google Drive
# from google.colab import drive; drive.mount("/content/drive"); !cp lits_run.zip /content/drive/MyDrive/